# Kalibratielijn stap voor stap

In deze notebook leer je stap voor stap hoe je een kalibratielijn maakt en interpreteert. Je hoeft niet alles over statistiek te weten: de belangrijkste begrippen worden onderweg rustig uitgelegd.

Een kalibratielijn gebruik je wanneer je een onbekende concentratie wilt bepalen uit een meetsignaal. Je meet eerst een aantal oplossingen waarvan je de concentratie al kent. Daarna zoek je het verband tussen:

- **x**: de bekende concentratie;
- **y**: de gemeten respons, bijvoorbeeld absorptie, piekoppervlak of signaalhoogte.

Als het verband ongeveer rechtlijnig is, gebruiken we een rechte lijn:

$$
\hat{y} = a + b x
$$

Daarin betekent:

- $\hat{y}$: de voorspelde respons volgens het model;
- $a$: het intercept, ook wel de asafsnede;
- $b$: de helling;
- $x$: de concentratie.

Aan het einde kun je de formule ook omdraaien. Bij een onbekende respons $y$ bereken je dan de concentratie:

$$
x = \frac{y-a}{b}
$$

## Leerdoelen

Na deze notebook kun je:

1. uitleggen wat een kalibratielijn is;
2. een lineaire regressie uitvoeren;
3. residuen interpreteren;
4. het verschil uitleggen tussen spreiding, standaarddeviatie en standaardfout;
5. betrouwbaarheidsintervallen voor intercept en helling interpreteren;
6. LOD en LOQ berekenen en uitleggen;
7. goodness-of-fit en lack-of-fit op hoofdlijnen begrijpen;
8. de berekeningen testen met controlegegevens.

## 0. Benodigdheden

We gebruiken een paar veelgebruikte Python-pakketten:

- `pandas` voor tabellen;
- `numpy` voor rekenen met arrays;
- `scipy.stats` voor verdelingen en statistische controletests;
- `matplotlib` voor grafieken.

Draai de volgende cel eerst.

In [ ]:
from dataclasses import dataclass
import math

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
pd.set_option("display.precision", 10)

## 1. Meetdata invoeren

We beginnen met een kleine dataset. Elke concentratie is twee keer gemeten. Dat heet een **duplometing**.

Waarom is dat handig?

Als je dezelfde concentratie twee keer meet, krijg je meestal niet exact dezelfde respons. Dat komt door normale meetspreiding. Duplometingen helpen om die meetspreiding zichtbaar te maken.

In de tabel hieronder zie je:

- `concentratie`: bekende concentratie van de standaard;
- `respons`: gemeten signaal.

In [ ]:
data = pd.DataFrame({
    "concentratie": [0.1, 0.1, 0.2, 0.2, 0.3, 0.3],
    "respons":      [0.101, 0.089, 0.212, 0.210, 0.312, 0.299],
})

data

### Eerste interpretatie

De respons wordt groter als de concentratie groter wordt. Dat is precies wat we hopen te zien bij een kalibratielijn.

Maar let op: de duplometingen zijn niet exact gelijk. Bij concentratie `0.1` zijn de responsen bijvoorbeeld `0.101` en `0.089`. Dat verschil is geen fout in de code; het is meetspreiding.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(data["concentratie"], data["respons"])
ax.set_xlabel("Concentratie")
ax.set_ylabel("Respons")
ax.set_title("Meetpunten voor de kalibratielijn")
ax.grid(True, alpha=0.3)
plt.show()

## 2. De statistische basis in gewone taal

Voordat we gaan rekenen, zetten we de belangrijkste begrippen op een rij.

### Gemiddelde

Het **gemiddelde** is de centrale waarde van een reeks getallen. Voor concentraties noemen we dit vaak $\bar{x}$. Voor responsen noemen we dit vaak $\bar{y}$.

Voorbeeld: als responsen rond `0.2` liggen, dan is het gemiddelde een soort middenpunt van die responsen.

### Afwijking van het gemiddelde

Een afwijking laat zien hoeveel een waarde boven of onder het gemiddelde ligt:

$$
x_i - \bar{x}
$$

Als de waarde groter is dan het gemiddelde, is de afwijking positief. Als de waarde kleiner is, is de afwijking negatief.

### Kwadraten

In regressie gebruiken we vaak kwadraten, zoals:

$$
(x_i - \bar{x})^2
$$

Waarom kwadrateren we?

1. negatieve en positieve afwijkingen heffen elkaar dan niet op;
2. grotere afwijkingen tellen zwaarder mee;
3. het maakt de wiskunde van lineaire regressie goed oplosbaar.

### Model en voorspelling

Een model is een vereenvoudigde beschrijving van de werkelijkheid. Hier gebruiken we een rechte lijn:

$$
\hat{y_i} = a + b x_i
$$

De echte meting noemen we $y_i$. De voorspelling van het model noemen we $\hat{y_i}$.

### Residu

Een **residu** is het verschil tussen wat je hebt gemeten en wat het model voorspelt:

$$
\text{residu}_i = y_i - \hat{y_i}
$$

Een residu dicht bij nul betekent dat het punt dicht bij de lijn ligt. Een groot residu betekent dat het punt verder van de lijn ligt.

## 3. Helperfuncties

De functies hieronder zorgen ervoor dat de rest van de notebook leesbaar blijft.

Je hoeft als student niet elk detail meteen te onthouden. Lees vooral de naam en de docstring boven elke functie:

- `fit_calibration_line`: zoekt de beste rechte lijn;
- `make_detail_table`: maakt tussenkolommen, zoals voorspellingen en residuen;
- `summarize_calibration`: berekent statistische kengetallen;
- `estimate_concentration`: rekent een onbekende respons terug naar concentratie;
- `plot_calibration`: tekent de meetpunten, kalibratielijn en de onder- en bovengrens.

In [ ]:
@dataclass
class CalibrationFit:
    """Resultaat van de rechte lijn y = intercept + slope * x."""
    intercept: float
    slope: float
    x_mean: float
    y_mean: float
    n: int
    sxx: float
    sum_x2: float
    correlation: float


def clean_calibration_data(df: pd.DataFrame) -> pd.DataFrame:
    """Controleer de kolommen en verwijder rijen zonder geldige concentratie/respons."""
    required = ["concentratie", "respons"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Ontbrekende kolommen: {missing}")

    cleaned = df[required].copy()
    cleaned["concentratie"] = pd.to_numeric(cleaned["concentratie"], errors="coerce")
    cleaned["respons"] = pd.to_numeric(cleaned["respons"], errors="coerce")
    cleaned = cleaned.dropna().reset_index(drop=True)

    if len(cleaned) < 3:
        raise ValueError("Gebruik minimaal drie meetpunten. Voor statistische spreiding zijn meer punten beter.")
    if cleaned["concentratie"].nunique() < 2:
        raise ValueError("Er zijn minimaal twee verschillende concentraties nodig.")
    return cleaned


def fit_calibration_line(df: pd.DataFrame) -> CalibrationFit:
    """Bereken de beste rechte lijn volgens de kleinste-kwadratenmethode.

    De kleinste-kwadratenmethode kiest de lijn waarbij de som van de kwadraten
    van de residuen zo klein mogelijk is.
    """
    cleaned = clean_calibration_data(df)
    x = cleaned["concentratie"].to_numpy(dtype=float)
    y = cleaned["respons"].to_numpy(dtype=float)

    x_mean = float(np.mean(x))
    y_mean = float(np.mean(y))
    sxx = float(np.sum((x - x_mean) ** 2))
    sum_x2 = float(np.sum(x ** 2))

    slope = float(np.sum((x - x_mean) * (y - y_mean)) / sxx)
    intercept = float(y_mean - slope * x_mean)
    correlation = float(np.corrcoef(x, y)[0, 1])

    return CalibrationFit(
        intercept=intercept,
        slope=slope,
        x_mean=x_mean,
        y_mean=y_mean,
        n=len(cleaned),
        sxx=sxx,
        sum_x2=sum_x2,
        correlation=correlation,
    )


def make_detail_table(df: pd.DataFrame, fit: CalibrationFit) -> pd.DataFrame:
    """Maak een tabel met voorspellingen, residuen en kwadraten.

    Deze tabel is handig voor onderwijs, omdat je kunt zien waar de
    samenvattende getallen vandaan komen.
    """
    detail = clean_calibration_data(df)
    x = detail["concentratie"].to_numpy(dtype=float)
    y = detail["respons"].to_numpy(dtype=float)
    y_hat = fit.intercept + fit.slope * x
    residual = y - y_hat

    detail["voorspelde_respons"] = y_hat
    detail["residu"] = residual
    detail["residu_kwadraat"] = residual ** 2
    detail["x_min_xgem"] = x - fit.x_mean
    detail["x_min_xgem_kwadraat"] = (x - fit.x_mean) ** 2
    detail["yhat_min_ygem_kwadraat"] = (y_hat - fit.y_mean) ** 2
    detail["y_min_ygem_kwadraat"] = (y - fit.y_mean) ** 2
    return detail


def summarize_calibration(df: pd.DataFrame, alpha: float = 0.05) -> tuple[pd.Series, pd.DataFrame]:
    """Bereken de belangrijkste kengetallen voor de kalibratielijn."""
    fit = fit_calibration_line(df)
    detail = make_detail_table(df, fit)

    n = fit.n
    df_residual = n - 2
    ss_residual = float(detail["residu_kwadraat"].sum())
    ss_regression = float(detail["yhat_min_ygem_kwadraat"].sum())
    ss_total = float(detail["y_min_ygem_kwadraat"].sum())

    ms_residual = ss_residual / df_residual
    syx = math.sqrt(ms_residual)

    # Standaardfouten van intercept en helling
    se_slope = syx / math.sqrt(fit.sxx)
    se_intercept = syx * math.sqrt(1 / n + fit.x_mean**2 / fit.sxx)

    t_critical = float(stats.t.ppf(1 - alpha / 2, df_residual))
    slope_ci_half_width = t_critical * se_slope
    intercept_ci_half_width = t_critical * se_intercept

    # Determinatiecoëfficiënt: welk deel van de y-spreiding wordt door de lijn verklaard?
    r_squared = 1 - ss_residual / ss_total

    # Detectie- en kwantificatielimiet volgens de veelgebruikte benadering 3*Sy/x/b en 10*Sy/x/b
    lod = 3 * syx / fit.slope if not np.isclose(fit.slope, 0.0) else np.nan
    loq = 10 * syx / fit.slope if not np.isclose(fit.slope, 0.0) else np.nan

    # Goodness-of-fit F-toets voor de regressielijn als geheel
    df_regression = 1
    ms_regression = ss_regression / df_regression
    f_goodness = np.inf if np.isclose(ms_residual, 0.0) else ms_regression / ms_residual
    f_goodness_critical = float(stats.f.ppf(0.95, df_regression, df_residual))

    # Lack-of-fit: splits residuele spreiding in pure meetspreiding en modelafwijking
    lof = lack_of_fit_table(detail, fit)

    summary = pd.Series({
        "a_intercept": fit.intercept,
        "b_helling": fit.slope,
        "n_meetpunten": n,
        "vrijheidsgraden_residu": df_residual,
        "x_gemiddelde": fit.x_mean,
        "y_gemiddelde": fit.y_mean,
        "correlatie_r": fit.correlation,
        "r_kwadraat": r_squared,
        "SS_residu": ss_residual,
        "SS_regressie": ss_regression,
        "SS_totaal": ss_total,
        "MS_residu": ms_residual,
        "Sy_x": syx,
        "SE_intercept": se_intercept,
        "SE_helling": se_slope,
        "t_kritisch_95pct": t_critical,
        "intercept_CI_onder": fit.intercept - intercept_ci_half_width,
        "intercept_CI_boven": fit.intercept + intercept_ci_half_width,
        "helling_CI_onder": fit.slope - slope_ci_half_width,
        "helling_CI_boven": fit.slope + slope_ci_half_width,
        "LOD": lod,
        "LOQ": loq,
        "F_goodness_of_fit": f_goodness,
        "F_kritisch_goodness_of_fit": f_goodness_critical,
        "goodness_significant": f_goodness > f_goodness_critical,
        "SS_pure_error": lof.loc["pure_error", "SS"],
        "SS_lack_of_fit": lof.loc["lack_of_fit", "SS"],
        "F_lack_of_fit": lof.loc["lack_of_fit", "F"],
        "F_kritisch_lack_of_fit": lof.loc["lack_of_fit", "F_kritisch_95pct"],
        "lack_of_fit_significant": lof.loc["lack_of_fit", "F" ] > lof.loc["lack_of_fit", "F_kritisch_95pct"],
    })
    return summary, detail


def lack_of_fit_table(detail: pd.DataFrame, fit: CalibrationFit) -> pd.DataFrame:
    """Bereken pure error en lack-of-fit bij herhaalde concentraties.

    Pure error = spreiding tussen herhalingen binnen dezelfde concentratie.
    Lack-of-fit = afwijking van de groepsgemiddelden ten opzichte van de rechte lijn.
    """
    grouped = detail.groupby("concentratie", sort=True)
    number_of_groups = grouped.ngroups
    n = len(detail)

    # Pure error: binnen elke concentratie kijken we naar afwijking t.o.v. het groepsgemiddelde
    pure_error_ss = 0.0
    lack_of_fit_ss = 0.0
    for concentration, group in grouped:
        y_values = group["respons"].to_numpy(dtype=float)
        y_group_mean = float(np.mean(y_values))
        pure_error_ss += float(np.sum((y_values - y_group_mean) ** 2))

        yhat_group = fit.intercept + fit.slope * float(concentration)
        lack_of_fit_ss += len(group) * (y_group_mean - yhat_group) ** 2

    df_pure_error = n - number_of_groups
    df_lack_of_fit = number_of_groups - 2

    ms_pure_error = pure_error_ss / df_pure_error if df_pure_error > 0 else np.nan
    ms_lack_of_fit = lack_of_fit_ss / df_lack_of_fit if df_lack_of_fit > 0 else np.nan
    f_lack_of_fit = (np.inf if np.isclose(ms_pure_error, 0.0) else ms_lack_of_fit / ms_pure_error) if df_pure_error > 0 and df_lack_of_fit > 0 else np.nan
    f_critical = stats.f.ppf(0.95, df_lack_of_fit, df_pure_error) if df_pure_error > 0 and df_lack_of_fit > 0 else np.nan

    return pd.DataFrame({
        "SS": [pure_error_ss, lack_of_fit_ss, pure_error_ss + lack_of_fit_ss],
        "vrijheidsgraden": [df_pure_error, df_lack_of_fit, df_pure_error + df_lack_of_fit],
        "MS": [ms_pure_error, ms_lack_of_fit, np.nan],
        "F": [np.nan, f_lack_of_fit, np.nan],
        "F_kritisch_95pct": [np.nan, f_critical, np.nan],
    }, index=["pure_error", "lack_of_fit", "residu_totaal"])


def estimate_concentration(response: float, summary: pd.Series) -> float:
    """Schat een onbekende concentratie uit een gemeten respons."""
    return (response - summary["a_intercept"]) / summary["b_helling"]


def calibration_bounds(x_values: np.ndarray, summary: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    """Bereken eenvoudige onder- en bovengrenzen voor de kalibratielijn.

    Deze grenzen gebruiken het 95%-betrouwbaarheidsinterval van het intercept
    en het 95%-betrouwbaarheidsinterval van de helling.

    Let op: dit is een didactische, eenvoudige manier om onzekerheid rond de lijn
    zichtbaar te maken. Het is niet hetzelfde als een formele simultane
    betrouwbaarheidsband voor de hele lijn.
    """
    lower = summary["intercept_CI_onder"] + summary["helling_CI_onder"] * x_values
    upper = summary["intercept_CI_boven"] + summary["helling_CI_boven"] * x_values
    return lower, upper


def plot_calibration(df: pd.DataFrame, summary: pd.Series, title: str = "Kalibratielijn") -> None:
    """Teken meetpunten, de berekende kalibratielijn en eenvoudige grenzen."""
    cleaned = clean_calibration_data(df)
    x = cleaned["concentratie"].to_numpy(dtype=float)
    y = cleaned["respons"].to_numpy(dtype=float)

    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = summary["a_intercept"] + summary["b_helling"] * x_line
    y_lower, y_upper = calibration_bounds(x_line, summary)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(x, y, label="metingen")
    ax.plot(x_line, y_line, label="berekende lijn")
    ax.plot(x_line, y_lower, linestyle="--", label="ondergrens")
    ax.plot(x_line, y_upper, linestyle="--", label="bovengrens")
    ax.fill_between(x_line, y_lower, y_upper, alpha=0.15, label="gebied tussen grenzen")
    ax.set_xlabel("Concentratie")
    ax.set_ylabel("Respons")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

## 4. De kalibratielijn berekenen

We berekenen nu de rechte lijn die het beste bij de meetpunten past.

De formule is:

$$
\hat{y} = a + bx
$$

Voor onze data berekent Python dus twee getallen:

- $a$: intercept;
- $b$: helling.

### Wat betekent de helling?

De helling geeft aan hoeveel de respons toeneemt als de concentratie met 1 eenheid toeneemt.

Een grotere helling betekent dat de methode gevoeliger is: een kleine verandering in concentratie geeft dan een grotere verandering in respons.

### Wat betekent het intercept?

Het intercept is de voorspelde respons bij concentratie 0.

In een ideale wereld is dat soms 0, maar in echte metingen kan er een achtergrondsignaal zijn. Denk bijvoorbeeld aan blanco-respons, instrumentruis of matrixeffecten.

In [ ]:
summary, detail = summarize_calibration(data)
summary[["a_intercept", "b_helling", "x_gemiddelde", "y_gemiddelde"]]

In [ ]:
plot_calibration(data, summary)

### Onder- en bovengrens in de grafiek

De grafiek toont nu niet alleen de meetpunten en de berekende lijn, maar ook een **ondergrens** en **bovengrens**.

Deze grenzen laten zien dat de kalibratielijn niet één absoluut zekere lijn is. We schatten de lijn op basis van een beperkt aantal metingen. Daardoor is er onzekerheid in:

- het **intercept**: waar de lijn de y-as snijdt;
- de **helling**: hoe steil de lijn loopt.

De onder- en bovengrens worden hier berekend met de 95%-betrouwbaarheidsintervallen van intercept en helling:

$$
\text{ondergrens} = a_{onder} + b_{onder}x
$$

$$
\text{bovengrens} = a_{boven} + b_{boven}x
$$

In gewone taal: we tekenen een lage en een hoge lijn die passen bij de onzekerheid in de berekende kalibratielijn.

Belangrijk om te onthouden: dit is een eenvoudige en inzichtelijke manier om onzekerheid zichtbaar te maken. In gevorderde statistiek bestaan ook formelere betrouwbaarheidsbanden en voorspellingsintervallen. Voor deze notebook is vooral belangrijk dat je ziet dat een kalibratielijn altijd met onzekerheid komt.

## 5. Tussenstappen bekijken

Een goede manier om regressie te begrijpen is niet alleen naar het eindresultaat kijken, maar ook naar de tussenkolommen.

Belangrijke kolommen:

- `voorspelde_respons`: respons volgens de berekende lijn;
- `residu`: gemeten respons min voorspelde respons;
- `residu_kwadraat`: residu in het kwadraat;
- `yhat_min_ygem_kwadraat`: hoeveel de voorspelling afwijkt van het gemiddelde;
- `y_min_ygem_kwadraat`: hoeveel de meting afwijkt van het gemiddelde.

Deze kwadraten vormen de basis voor veel statistische termen in deze notebook.

In [ ]:
detail

## 6. Residuen begrijpen

Een residu is:

$$
\text{residu} = y - \hat{y}
$$

Dat betekent:

- positief residu: het meetpunt ligt boven de lijn;
- negatief residu: het meetpunt ligt onder de lijn;
- residu rond nul: het meetpunt ligt dicht bij de lijn.

Bij een goede lineaire kalibratie verwachten we dat residuen willekeurig rond nul liggen. We willen geen duidelijk patroon zien, zoals steeds positieve residuen bij lage concentraties en negatieve residuen bij hoge concentraties.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.axhline(0, linestyle="--")
ax.scatter(detail["concentratie"], detail["residu"])
ax.set_xlabel("Concentratie")
ax.set_ylabel("Residu")
ax.set_title("Residuenplot")
ax.grid(True, alpha=0.3)
plt.show()

### Hoe lees je een residuenplot?

Een residuenplot is een controleplaatje.

Goed teken:

- punten liggen willekeurig rond nul;
- geen duidelijke kromming;
- spreiding is ongeveer vergelijkbaar over het concentratiegebied.

Mogelijk probleem:

- duidelijke boogvorm: misschien is het verband niet lineair;
- spreiding wordt groter bij hogere concentratie: misschien is de meetfout niet constant;
- één punt ligt ver weg: mogelijk een uitschieter of invoerfout.

## 7. Som van kwadraten: SS

Veel statistiek in regressie gaat over **spreiding**. Die spreiding drukken we vaak uit als een som van kwadraten. In tabellen zie je daarvoor vaak de afkorting **SS**, van het Engelse *sum of squares*.

We gebruiken hier drie belangrijke sommen:

### 1. Totale spreiding: SS totaal

$$
SS_{totaal} = \sum (y_i - \bar{y})^2
$$

Dit is de totale spreiding van alle responswaarden rond hun gemiddelde.

### 2. Verklaarde spreiding: SS regressie

$$
SS_{regressie} = \sum (\hat{y_i} - \bar{y})^2
$$

Dit is de spreiding die door de regressielijn wordt verklaard.

### 3. Onverklaarde spreiding: SS residu

$$
SS_{residu} = \sum (y_i - \hat{y_i})^2
$$

Dit is wat overblijft: de afwijkingen tussen metingen en model.

Samen geldt:

$$
SS_{totaal} \approx SS_{regressie} + SS_{residu}
$$

In [ ]:
summary[["SS_totaal", "SS_regressie", "SS_residu"]]

## 8. Correlatie en R-kwadraat

### Correlatie, r

De correlatie $r$ ligt tussen -1 en +1.

- $r$ dicht bij +1: sterke stijgende rechte relatie;
- $r$ dicht bij -1: sterke dalende rechte relatie;
- $r$ rond 0: weinig lineair verband.

Bij een kalibratielijn hopen we meestal op een waarde dicht bij +1.

### R-kwadraat, $R^2$

$R^2$ geeft aan welk deel van de variatie in de respons door de lijn wordt verklaard.

Voorbeeld: $R^2 = 0.99$ betekent dat ongeveer 99% van de spreiding in de respons door het lineaire model wordt verklaard.

Belangrijk: een hoge $R^2$ is prettig, maar niet genoeg. Je moet ook residuen en lack-of-fit bekijken.

In [ ]:
summary[["correlatie_r", "r_kwadraat"]]

## 9. Vrijheidsgraden

Vrijheidsgraden klinken ingewikkeld, maar het idee is eenvoudig:

> Vrijheidsgraden zijn het aantal onafhankelijke stukjes informatie dat nog vrij kan variëren nadat je parameters hebt geschat.

Bij een rechte lijn schatten we twee parameters:

- intercept $a$;
- helling $b$.

Als we $n$ meetpunten hebben, blijven er voor de residuen daarom over:

$$
\text{vrijheidsgraden residu} = n - 2
$$

Waarom min 2? Omdat twee vrijheden zijn gebruikt om de lijn te bepalen.

In [ ]:
summary[["n_meetpunten", "vrijheidsgraden_residu"]]

## 10. Sy/x: standaardafwijking rond de regressielijn

$S_{y/x}$ spreek je vaak uit als: *S y gegeven x*.

Het is een maat voor de typische verticale spreiding van de meetpunten rond de kalibratielijn.

De formule is:

$$
S_{y/x} = \sqrt{\frac{SS_{residu}}{n-2}}
$$

Interpretatie:

- kleine $S_{y/x}$: punten liggen dicht bij de lijn;
- grote $S_{y/x}$: punten liggen verder van de lijn.

$S_{y/x}$ heeft dezelfde eenheid als de respons. Als de respons bijvoorbeeld absorptie is, dan is $S_{y/x}$ ook in absorptie-eenheden.

In [ ]:
summary[["SS_residu", "MS_residu", "Sy_x"]]

## 11. Standaardfout van intercept en helling

We hebben één dataset gemeten. Maar stel dat we het experiment morgen opnieuw doen. Dan krijgen we waarschijnlijk net iets andere meetwaarden, en dus ook net een andere lijn.

De **standaardfout** beschrijft hoe onzeker een geschatte parameter is.

Hier gebruiken we:

- `SE_intercept`: onzekerheid van het intercept;
- `SE_helling`: onzekerheid van de helling.

Belangrijk verschil:

- standaarddeviatie: spreiding van metingen;
- standaardfout: onzekerheid van een geschatte parameter, zoals een gemiddelde, intercept of helling.

In [ ]:
summary[["SE_intercept", "SE_helling"]]

## 12. Betrouwbaarheidsinterval

Een **95%-betrouwbaarheidsinterval** geeft een bereik van waarden die passen bij de data en de gekozen onzekerheid.

Voor de helling betekent dit bijvoorbeeld:

> Op basis van deze metingen ligt de echte helling waarschijnlijk ergens binnen dit interval.

De algemene vorm is:

$$
\text{schatting} \pm t_{kritisch} \times \text{standaardfout}
$$

De factor $t_{kritisch}$ komt uit de t-verdeling. Bij weinig meetpunten is die factor groter, omdat de onzekerheid groter is.

Let op: een 95%-betrouwbaarheidsinterval betekent niet dat er 95% kans is dat deze specifieke berekende onder- en bovengrens de echte waarde bevatten. Strikt genomen betekent het: als je dit experiment heel vaak zou herhalen en telkens zo'n interval zou maken, dan bevat ongeveer 95% van die intervallen de echte waarde.

In [ ]:
summary[[
    "t_kritisch_95pct",
    "intercept_CI_onder", "a_intercept", "intercept_CI_boven",
    "helling_CI_onder", "b_helling", "helling_CI_boven",
]]

## 13. LOD en LOQ

### LOD: limit of detection

De **LOD** is de detectielimiet. Dit is een schatting van de laagste concentratie waarbij je kunt zeggen: er is waarschijnlijk iets aanwezig.

Een veelgebruikte benadering is:

$$
LOD = \frac{3 S_{y/x}}{b}
$$

### LOQ: limit of quantification

De **LOQ** is de kwantificatielimiet. Dit is een schatting van de laagste concentratie waarbij je niet alleen aanwezigheid detecteert, maar ook redelijk kwantitatief kunt meten.

Een veelgebruikte benadering is:

$$
LOQ = \frac{10 S_{y/x}}{b}
$$

### Waarom delen door de helling?

$S_{y/x}$ is spreiding in respons-eenheden. Maar LOD en LOQ willen we in concentratie-eenheden. De helling $b$ vertelt hoeveel respons verandert per concentratie-eenheid. Daarom gebruiken we de helling om respons-spreiding om te rekenen naar concentratie-spreiding.

### Waarom 3 en 10?

De factoren 3 en 10 zijn praktische conventies. Ze worden vaak gebruikt als eenvoudige vuistregel. In echte validatieprotocollen kunnen andere definities of strengere regels gelden.

In [ ]:
summary[["Sy_x", "b_helling", "LOD", "LOQ"]]

## 14. Goodness-of-fit

**Goodness-of-fit** betekent letterlijk: hoe goed past het model bij de data?

Hier gebruiken we een F-toets die kijkt of de regressielijn als geheel een duidelijke verklaring geeft voor de variatie in de respons.

De gedachte is:

- verklaarde spreiding door de lijn: `SS_regressie`;
- onverklaarde spreiding rond de lijn: `SS_residu`.

De F-waarde is grofweg:

$$
F = \frac{MS_{regressie}}{MS_{residu}}
$$

Waarbij MS staat voor *mean square*, oftewel som van kwadraten gedeeld door vrijheidsgraden.

Als de F-waarde groter is dan de kritische F-waarde, is de regressie statistisch significant. Dat betekent: de rechte lijn verklaart duidelijk meer dan je op basis van willekeurige spreiding zou verwachten.

In eenvoudige taal: er is dan een duidelijk lineair verband tussen concentratie en respons.

In [ ]:
summary[["F_goodness_of_fit", "F_kritisch_goodness_of_fit", "goodness_significant"]]

## 15. Lack-of-fit

Een hoge correlatie of significante regressie betekent nog niet automatisch dat een rechte lijn het beste model is. Misschien is het verband licht krom, maar lijkt het met weinig punten toch bijna recht.

Daarvoor gebruiken we de **lack-of-fit** toets.

### Het idee

Omdat we duplometingen hebben, kunnen we de residuele spreiding splitsen in twee delen:

1. **Pure error**: normale meetspreiding tussen herhalingen bij dezelfde concentratie.
2. **Lack-of-fit**: extra afwijking doordat de gekozen rechte lijn misschien niet goed bij de gemiddelden past.

Als lack-of-fit groot is ten opzichte van pure error, kan dat betekenen dat het lineaire model niet passend is.

### Interpretatie

- `lack_of_fit_significant = False`: geen duidelijk bewijs dat de rechte lijn ongeschikt is;
- `lack_of_fit_significant = True`: er is aanwijzing dat de rechte lijn mogelijk niet goed past.

Let op: met heel weinig meetpunten moet je voorzichtig zijn. Een toets is een hulpmiddel, geen automatische waarheid.

In [ ]:
fit = fit_calibration_line(data)
lof_table = lack_of_fit_table(detail, fit)
lof_table

In [ ]:
summary[["F_lack_of_fit", "F_kritisch_lack_of_fit", "lack_of_fit_significant"]]

## 16. Onbekende concentratie berekenen

Stel dat je een onbekend monster meet en de respons is `0.250`.

We gebruiken dan de omgekeerde kalibratieformule:

$$
x = \frac{y-a}{b}
$$

Daarbij is:

- $y$: respons van het onbekende monster;
- $a$: intercept;
- $b$: helling.

In [ ]:
onbekende_respons = 0.250
concentratie_schatting = estimate_concentration(onbekende_respons, summary)
concentratie_schatting

### Belangrijke waarschuwing bij onbekende monsters

Gebruik de kalibratielijn bij voorkeur alleen binnen het gemeten concentratiegebied.

In deze dataset loopt het concentratiegebied van 0.1 tot 0.3. Een respons die overeenkomt met bijvoorbeeld concentratie 0.8 zou buiten het kalibratiegebied vallen. Dat heet **extrapolatie** en is minder betrouwbaar.

In [ ]:
concentratie_min = data["concentratie"].min()
concentratie_max = data["concentratie"].max()

print(f"Kalibratiegebied: {concentratie_min:.3f} t/m {concentratie_max:.3f}")
print(f"Geschatte concentratie onbekend monster: {concentratie_schatting:.3f}")

if concentratie_min <= concentratie_schatting <= concentratie_max:
    print("Deze schatting ligt binnen het kalibratiegebied.")
else:
    print("Deze schatting ligt buiten het kalibratiegebied: wees voorzichtig met interpretatie.")

## 17. Automatische controles

Een notebook voor onderwijs moet niet alleen rekenen, maar ook controleren of de functies logisch werken.

We doen drie controles:

1. Een perfecte rechte lijn moet exact de juiste helling en intercept geven.
2. De regressie-uitkomst moet overeenkomen met een onafhankelijke berekening via `numpy.polyfit`.
3. De regressie-uitkomst moet overeenkomen met `scipy.stats.linregress`.

Als alle controles slagen, geeft Python geen foutmelding.

In [ ]:
# Test 1: perfecte rechte lijn y = 1 + 2x
perfect_data = pd.DataFrame({
    "concentratie": [0, 1, 2, 3, 4],
    "respons":      [1, 3, 5, 7, 9],
})
perfect_summary, perfect_detail = summarize_calibration(perfect_data)

assert np.isclose(perfect_summary["a_intercept"], 1.0)
assert np.isclose(perfect_summary["b_helling"], 2.0)
assert np.isclose(perfect_summary["SS_residu"], 0.0)
assert np.isclose(perfect_summary["r_kwadraat"], 1.0)

# Test 2: controle met numpy.polyfit op de echte onderwijsdata
coef = np.polyfit(data["concentratie"], data["respons"], deg=1)
np_slope, np_intercept = coef[0], coef[1]
assert np.isclose(summary["a_intercept"], np_intercept)
assert np.isclose(summary["b_helling"], np_slope)

# Test 3: controle met scipy.stats.linregress
linreg = stats.linregress(data["concentratie"], data["respons"])
assert np.isclose(summary["a_intercept"], linreg.intercept)
assert np.isclose(summary["b_helling"], linreg.slope)
assert np.isclose(summary["correlatie_r"], linreg.rvalue)

print("Alle controles zijn geslaagd.")

## 18. Oefening voor studenten

Pas de dataset hieronder aan en draai de cellen opnieuw.

Vragen om te beantwoorden:

1. Wat gebeurt er met de helling als de respons sneller stijgt?
2. Wat gebeurt er met $S_{y/x}$ als je meer meetruis toevoegt?
3. Wat gebeurt er met LOD en LOQ als de spreiding groter wordt?
4. Zie je in de residuenplot een patroon?
5. Is de lack-of-fit significant?

Tip: verander eerst maar één of twee responswaarden, zodat je goed ziet welk effect dat heeft.

In [ ]:
oefen_data = pd.DataFrame({
    "concentratie": [0.1, 0.1, 0.2, 0.2, 0.3, 0.3, 0.4, 0.4],
    "respons":      [0.11, 0.10, 0.20, 0.22, 0.31, 0.29, 0.40, 0.43],
})

oefen_summary, oefen_detail = summarize_calibration(oefen_data)
oefen_summary[[
    "a_intercept", "b_helling", "r_kwadraat", "Sy_x", "LOD", "LOQ",
    "lack_of_fit_significant"
]]

In [ ]:
plot_calibration(oefen_data, oefen_summary, title="Kalibratielijn voor oefendata")

In [ ]:
oefen_detail

## 19. Samenvatting

In deze notebook heb je een volledige kalibratieberekening uitgevoerd.

De belangrijkste begrippen:

- **Kalibratielijn**: rechte lijn die bekende concentraties koppelt aan gemeten responsen.
- **Intercept**: voorspelde respons bij concentratie 0.
- **Helling**: gevoeligheid van de methode; responsverandering per concentratie-eenheid.
- **Residu**: verschil tussen meting en model.
- **SS**: som van kwadraten; maat voor spreiding.
- **Vrijheidsgraden**: hoeveel onafhankelijke informatie overblijft na het schatten van parameters.
- **$S_{y/x}$**: typische spreiding van punten rond de regressielijn.
- **Standaardfout**: onzekerheid van een geschatte parameter.
- **Betrouwbaarheidsinterval**: bereik van plausibele waarden voor een parameter.
- **LOD**: geschatte laagste detecteerbare concentratie.
- **LOQ**: geschatte laagste kwantificeerbare concentratie.
- **Goodness-of-fit**: toets of de regressie als geheel duidelijk is.
- **Lack-of-fit**: toets of een rechte lijn wel passend is ten opzichte van de meetspreiding.

De belangrijkste praktische les:

> Kijk nooit alleen naar de formule van de lijn. Bekijk ook de residuen, de spreiding, de onzekerheid en de grenzen van het kalibratiegebied.